# Fase 3 — Ingeniería de características

**Objetivo:** convertir los datos limpios en la **matriz numérica final** que consumirán los modelos.

Un modelo de machine learning no "entiende" pasajeros: solo ve una tabla de números y busca patrones
en ellos. La **ingeniería de características** (*feature engineering*) consiste en transformar los
datos crudos para **ponerle la señal en bandeja** al modelo:

- **Crear** variables que expresen directamente lo que descubrimos en el EDA
  (p. ej. `FamilySize`, porque la supervivencia depende del tamaño de la familia).
- **Codificar** el texto como números **sin inventar información falsa** (aquí vive el
  *one-hot encoding* que veremos en la sección 4).
- **Eliminar** lo ya explotado o inútil, para no meter ruido.

En Titanic esta fase importa más que la elección del modelo: la diferencia entre un 0.76 y un 0.78+
en el leaderboard suele estar aquí, no en probar veinte algoritmos.

Partimos de los datos limpios de la Fase 2 (`data/processed/`), que ya traen `Title` y `HasCabin`.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append("../src")

train = pd.read_csv("../data/processed/train_limpio.csv")
test  = pd.read_csv("../data/processed/test_limpio.csv")
print("train:", train.shape, "| test:", test.shape)
print("nulos:", train.isna().sum().sum() + test.isna().sum().sum(), "(deben ser 0, veniamos de la Fase 2)")
train.head(3)

train: (891, 13) | test: (418, 12)
nulos: 0 (deben ser 0, veniamos de la Fase 2)


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Embarked,Title,HasCabin
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,S,Mr,0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C,Mrs,1
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,S,Miss,0


## 1 · `FamilySize` e `IsAlone`

En el EDA vimos que la supervivencia según la familia tiene forma de **U invertida**: solos ~30%,
familias de 2-4 ~55-72%, de 5+ ~16%. Pero esa señal está *repartida* entre dos columnas (`SibSp` y
`Parch`) que por separado dicen poco. Las combinamos:

```
FamilySize = SibSp + Parch + 1     (el +1 es el propio pasajero)
IsAlone    = 1 si FamilySize == 1
```

¿Por qué añadir `IsAlone` si ya está implícita en `FamilySize`? Porque le da al modelo el "escalón"
más importante (solo vs acompañado) ya construido. A un modelo lineal, que solo puede aprender
*"más familia → más supervivencia"* o lo contrario, la U invertida se le escapa; `IsAlone` le
rescata al menos la mitad de esa señal. A los árboles les ahorra trabajo.

In [2]:
for df in (train, test):
    df["FamilySize"] = df["SibSp"] + df["Parch"] + 1
    df["IsAlone"] = (df["FamilySize"] == 1).astype(int)

print(train.groupby("IsAlone")["Survived"].agg(["count", "mean"]).round(3))

         count   mean
IsAlone              
0          354  0.506
1          537  0.304


## 2 · `TicketGroup`: contar quién viaja junto (y un matiz importante)

`FamilySize` no ve a los grupos que no eran familia: amigos, criados, niñeras. Los billetes
compartidos sí: contamos cuántos pasajeros llevan el mismo número de billete.

**El matiz:** ¿contamos solo dentro de train? Mira lo que pasa con la familia Sage:

In [3]:
completo = pd.concat([train, test])           # los 1309 pasajeros del dataset

sage = completo[completo["Ticket"] == "CA. 2343"]
print(sage[["Name", "Age", "FamilySize", "Survived"]].to_string())
print("\nEn train:", train["Ticket"].eq("CA. 2343").sum(), "| en test:", test["Ticket"].eq("CA. 2343").sum())
print("Tickets con miembros repartidos entre train y test:",
      len(set(train["Ticket"]) & set(test["Ticket"])))

                                  Name   Age  FamilySize  Survived
159         Sage, Master. Thomas Henry   4.0          11       0.0
180       Sage, Miss. Constance Gladys  18.0          11       0.0
201                Sage, Mr. Frederick  26.0          11       0.0
324           Sage, Mr. George John Jr  26.0          11       0.0
792            Sage, Miss. Stella Anna  18.0          11       0.0
846           Sage, Mr. Douglas Bullen  26.0          11       0.0
863  Sage, Miss. Dorothy Edith "Dolly"  18.0          11       0.0
188                    Sage, Miss. Ada  18.0          11       NaN
342              Sage, Mr. John George  26.0          11       NaN
360        Sage, Master. William Henry  14.5          11       NaN
365     Sage, Mrs. John (Annie Bullen)  31.0          11       NaN

En train: 7 | en test: 4
Tickets con miembros repartidos entre train y test: 115


Los Sage eran **11** (fíjate: `FamilySize` = 8+2+1 = 11 lo confirma, y las filas de test tienen
`Survived = NaN` porque es lo que habrá que predecir). Pero 7 cayeron en train y 4 en test: contando
solo train diríamos "grupo de 7". Hay **115 billetes** así de repartidos.

**Decisión:** contar sobre train + test juntos. ¿No era eso *leakage*? No, y la distinción es
importante de entender:

- **Leakage** = usar información del **target** (`Survived`) o estadísticas que no tendrías al
  predecir. Las medianas de la Fase 2 eran eso: números que resumen el train y que el test no debe
  tocar.
- El tamaño del grupo **no usa el target para nada**: es un hecho de la lista de pasajeros, y la
  lista completa la conocemos en el momento de predecir (Kaggle nos la da — es el propio test.csv).
  La familia Sage son 11 personas, caigan sus filas donde caigan.

De regalo, `TicketGroup` nos da la feature que arregla `Fare`: recuerda que el fare es **por
billete**, no por persona — los 11 Sage "pagaron" 69.55£ cada uno según la columna, pero en realidad
son 69.55£ **entre once**. De ahí:

```
FarePerPerson = Fare / TicketGroup
```

In [4]:
conteo_tickets = completo["Ticket"].value_counts()

for df in (train, test):
    df["TicketGroup"] = df["Ticket"].map(conteo_tickets)
    df["FarePerPerson"] = df["Fare"] / df["TicketGroup"]

print("Supervivencia por tamano de grupo de billete (train):")
print(train.assign(TG=train["TicketGroup"].clip(upper=5)).groupby("TG")["Survived"].agg(["count", "mean"]).round(3))
print("\nEjemplo Sage: Fare =", train.loc[train.Ticket == "CA. 2343", "Fare"].iloc[0],
      "-> FarePerPerson =", train.loc[train.Ticket == "CA. 2343", "FarePerPerson"].round(2).iloc[0])

Supervivencia por tamano de grupo de billete (train):
    count   mean
TG              
1     481  0.270
2     181  0.514
3     101  0.653
4      44  0.727
5      84  0.250

Ejemplo Sage: Fare = 69.55 -> FarePerPerson = 6.32


La mediana de fare de 3ª clase era 8.05£ — los Sage, con sus "69.55£" aparentes de billete caro,
pagaron en realidad **6.32£ por cabeza**, precio típico de 3ª. `FarePerPerson` cuenta la verdad que
`Fare` disfrazaba.

(Nota: 17 pasajeros tienen `Fare = 0` — empleados de la naviera, en su mayoría. Su `FarePerPerson`
queda en 0, lo cual es información válida, no un error.)

## 3 · ¿Y las bandas de edad?

En el plan aparecían como opcionales (`AgeBand`, `FareBand`). Decisión: **no las creamos**. Los
modelos de árboles que usaremos parten las variables continuas donde mejor les convenga
("¿Age ≤ 6.5?") — trocearlas nosotros a mano solo les quitaría resolución. Si en la Fase 5 usáramos
intensivamente modelos lineales, lo reconsideraríamos.

## 4 · Codificar las categóricas: el one-hot encoding

Nos quedan tres columnas de texto: `Sex`, `Embarked`, `Title`. Los modelos necesitan números, pero
**cómo** conviertas el texto importa muchísimo. La tentación ingenua es numerar las categorías:
`S=0, C=1, Q=2`. El problema: eso le dice al modelo que `Q > C > S` y que "C está a mitad de camino
entre S y Q" — **un orden y unas distancias que no existen**. El modelo intentaría aprender de esa
geometría inventada.

La solución es el **one-hot encoding**: una columna binaria por categoría.

| Embarked | → | Embarked_S | Embarked_C | Embarked_Q |
|---|---|---|---|---|
| S | → | 1 | 0 | 0 |
| C | → | 0 | 1 | 0 |
| Q | → | 0 | 0 | 1 |

Ahora cada categoría es independiente: no hay orden falso. Reglas de decisión que aplicamos:

- **`Sex`**: binaria → basta **una** columna (`IsFemale` 0/1). Crear dos (`Sex_male`, `Sex_female`)
  sería redundante: una es exactamente 1 menos la otra.
- **`Embarked`, `Title`**: nominales → one-hot completo.
- **`Pclass`**: ¡NO se codifica! Es **ordinal de verdad** (1ª > 2ª > 3ª es un orden real de estatus);
  el número ya representa bien esa información.
- **Vocabulario fijo**: declaramos las categorías posibles ANTES de codificar
  (`pd.Categorical(..., categories=[...])`). Así train y test producen exactamente las mismas
  columnas aunque a uno le faltara alguna categoría — y el orden de las columnas queda determinista.

> Detalle para más adelante (*dummy variable trap*): las 3 columnas de `Embarked` suman siempre 1,
> es decir, una es deducible de las otras dos. A los árboles les da igual; a una regresión lineal
> clásica le molestaría (colinealidad), pero nuestra logística llevará regularización, que lo
> gestiona sin problema. Por eso conservamos todas.

Todo esto (secciones 1, 2 y 4) está empaquetado en [`src/features.py`](../src/features.py), otra vez
como funciones reutilizables. Repetimos ahora el proceso completo desde cero usando el módulo:

In [5]:
import features

# Partimos de los datos limpios originales (sin las columnas que creamos a mano arriba)
train_limpio = pd.read_csv("../data/processed/train_limpio.csv")
test_limpio  = pd.read_csv("../data/processed/test_limpio.csv")

conteo = features.contar_grupos_ticket(train_limpio, test_limpio)
train_feat = features.construir(train_limpio, conteo)
test_feat  = features.construir(test_limpio, conteo)

print("train_feat:", train_feat.shape, "| test_feat:", test_feat.shape)
train_feat.head()

train_feat: (891, 19) | test_feat: (418, 18)


,PassengerId,Survived,Pclass,Age,Fare,HasCabin,FamilySize,IsAlone,TicketGroup,FarePerPerson,IsFemale,Embarked_S,Embarked_C,Embarked_Q,Title_Mr,Title_Mrs,Title_Miss,Title_Master,Title_Rare
0,1,0,3,22.0,7.2500,0,2,0,1,7.25000,0,1,0,0,1,0,0,0,0
1,2,1,1,38.0,71.2833,1,2,0,2,35.64165,1,0,1,0,0,1,0,0,0
2,3,1,3,26.0,7.9250,0,1,1,1,7.92500,1,1,0,0,0,0,1,0,0
3,4,1,1,35.0,53.1000,1,2,0,2,26.55000,1,1,0,0,0,1,0,0,0
4,5,0,3,35.0,8.0500,0,1,1,1,8.05000,0,1,0,0,1,0,0,0,0


## 5 · Verificación

Cuatro controles antes de dar la matriz por buena:

1. **Mismas columnas** en train y test (salvo `Survived`, que el test no tiene) y en el mismo orden.
2. **Todo numérico**: ni una columna de texto superviviente.
3. **Cero nulos** (las features nuevas podrían haberlos introducido — p. ej. un ticket sin conteo).
4. **Vistazo de cordura**: rangos razonables en las columnas nuevas.

In [6]:
cols_train = [c for c in train_feat.columns if c != "Survived"]
cols_test  = list(test_feat.columns)
print("1. mismas columnas y mismo orden:", cols_train == cols_test)

no_numericas = train_feat.select_dtypes(exclude="number").columns.tolist()
print("2. columnas no numericas:", no_numericas if no_numericas else "ninguna")

print("3. nulos: train", train_feat.isna().sum().sum(), "| test", test_feat.isna().sum().sum())

print("\n4. resumen de la matriz final (train):")
print(train_feat.drop(columns=["PassengerId", "Survived"]).describe().T[["min", "max", "mean"]].round(2))

1. mismas columnas y mismo orden: True
2. columnas no numericas: ninguna
3. nulos: train 0 | test 0

4. resumen de la matriz final (train):


                min     max   mean
Pclass         1.00    3.00   2.31
Age            0.42   80.00  29.14
Fare           0.00  512.33  32.20
HasCabin       0.00    1.00   0.23
FamilySize     1.00   11.00   1.90
IsAlone        0.00    1.00   0.60
TicketGroup    1.00   11.00   2.12
FarePerPerson  0.00  128.08  14.55
IsFemale       0.00    1.00   0.35
Embarked_S     0.00    1.00   0.73
Embarked_C     0.00    1.00   0.19
Embarked_Q     0.00    1.00   0.09
Title_Mr       0.00    1.00   0.58
Title_Mrs      0.00    1.00   0.14
Title_Miss     0.00    1.00   0.20
Title_Master   0.00    1.00   0.04
Title_Rare     0.00    1.00   0.03

Los cuatro controles pasan. La matriz final tiene **16 features** + `PassengerId` + `Survived`:

| Grupo | Features |
|---|---|
| Originales que sobreviven | `Pclass`, `Age`, `Fare` |
| De la Fase 2 | `HasCabin` |
| Familia y grupos | `FamilySize`, `IsAlone`, `TicketGroup`, `FarePerPerson` |
| Codificadas | `IsFemale`, `Embarked_S/C/Q`, `Title_Mr/Mrs/Miss/Master/Rare` |

⚠️ **`PassengerId` sigue en la tabla a propósito**: lo necesitaremos para construir el archivo de
submission (Kaggle pide `PassengerId, Survived`). Pero es un número de fila sin información — en la
Fase 4 habrá que **excluirlo explícitamente** de las features del modelo. Lo mismo `Survived`, claro:
es la respuesta, no una feature. Apúntalo: primer paso de la Fase 4 será separar `X` (features) de
`y` (target).

## 6 · ¿Las features nuevas llevan señal? Vistazo rápido

Antes de guardar, comprobemos que el esfuerzo valió la pena — correlación de cada feature con
`Survived` (con la advertencia de siempre: es una medida lineal, infravalora las señales con forma
de U como `FamilySize`):

In [7]:
correlaciones = (train_feat.drop(columns=["PassengerId"])
                 .corr()["Survived"]
                 .drop("Survived")
                 .sort_values(key=abs, ascending=False))
print(correlaciones.round(3).to_string())

Title_Mr        -0.549
IsFemale         0.543
Title_Mrs        0.339
Pclass          -0.338
Title_Miss       0.327
HasCabin         0.317
FarePerPerson    0.288
Fare             0.257
IsAlone         -0.203
Embarked_C       0.168
Embarked_S      -0.150
Title_Master     0.085
TicketGroup      0.065
Age             -0.061
Title_Rare       0.022
FamilySize       0.017
Embarked_Q       0.004


`IsFemale` (+0.54) y `Title_Mr` (-0.55) dominan, como esperábamos. De las creadas hoy:
`FarePerPerson` supera a `Fare` (nuestra corrección funcionó), e `IsAlone` (-0.20) aporta la parte
"escalón" de la señal de familia. `FamilySize` y `TicketGroup` salen bajas *en correlación lineal*,
pero ya sabemos por qué — su señal es la U invertida, y serán los árboles quienes la exploten.

## 7 · Guardar y cerrar la fase

In [8]:
train_feat.to_csv("../data/processed/train_features.csv", index=False)
test_feat.to_csv("../data/processed/test_features.csv", index=False)
import os
print("Guardado en data/processed/:", sorted(os.listdir("../data/processed")))

Guardado en data/processed/:

 ['test_features.csv', 'test_limpio.csv', 'train_features.csv', 'train_limpio.csv']


## 8 · Conclusiones de la Fase 3

- La matriz final tiene **16 features numéricas, sin nulos y con columnas idénticas** en train y test.
- Creamos 4 features nuevas guiadas por el EDA: `FamilySize`, `IsAlone`, `TicketGroup` (contado
  sobre los 1309 pasajeros — y entendimos por qué eso no es leakage), y `FarePerPerson` (que
  destapa el precio real que `Fare` escondía: los Sage no pagaron 69£, sino 6.32£ por cabeza).
- Codificamos sin inventar información: one-hot para nominales (`Embarked`, `Title`), una sola
  binaria para `Sex`, y `Pclass` intacta por ser ordinal genuina.
- Todo reproducible en 3 líneas vía [`src/features.py`](../src/features.py).

**Siguiente — Fase 4:** separar `X` e `y`, montar los dos baselines (género y regresión logística) y
—la pieza más importante de todo el proyecto— definir el protocolo de **validación cruzada** que
usaremos para juzgar honestamente cada modelo sin gastar submissions de Kaggle.